<a href="https://colab.research.google.com/github/Doan-Truc/Doan-Truc/blob/main/preprocessing-prethe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import json
from collections import Counter
from scipy.signal import butter, filtfilt, welch
from sklearn.decomposition import FastICA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report
import os


In [ ]:
fs = 128  # Sampling rate
run_names = ['Run1', 'Run2', 'Run3', 'Run4', 'Run5','Run6','Run7','Run8']

# Lấy đúng các nhãn hành động MI
valid_labels = ['3', '4', '5', '6']
label_map = {
    '3': 'Right Hand',
    '4': 'Left Hand',
    '5': 'Right Foot',
    '6': 'Left Foot'
}


Bandpass, ICA, Chuẩn hoá

In [2]:
import os
import json
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA

# ==== CONFIG ====
fs = 128  # Sampling rate (Hz)
run_names = ['Run1', 'Run2', 'Run3', 'Run4', 'Run5', 'Run6', 'Run7', 'Run8']

base_path = "/content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024"
output_dir = os.path.join(base_path, "cleaned_full_runs")
os.makedirs(output_dir, exist_ok=True)

# === Label mapping (motor imagery actions only) ===
label_map = {
    '1': 'Rest',
    '2': 'Cue',
    '3': 'Right Hand',
    '4': 'Left Hand',
    '5': 'Right Foot',
    '6': 'Left Foot'
}

# ==== FILTER FUNCTIONS ====
def bandpass_filter(eeg, low=8, high=30, fs=128, order=4):
    """Lọc EEG giữ dải tần 8–30 Hz"""
    b, a = butter(order, [low / (fs / 2), high / (fs / 2)], btype='band')
    return filtfilt(b, a, eeg, axis=0)

def apply_ica(eeg, n_components=None):
    """Giảm nhiễu bằng ICA"""
    ica = FastICA(n_components=n_components or eeg.shape[1], random_state=42)
    return ica.fit_transform(eeg)

def normalize(eeg):
    """Chuẩn hoá dữ liệu EEG (z-score)"""
    scaler = StandardScaler()
    return scaler.fit_transform(eeg)

# ==== MAIN PIPELINE ====
for run in run_names:
    print(f"\n🧠 Processing {run} ...")

    # === Load files ===
    csv_file = os.path.join(base_path, f"{run}_Raw Signals.csv")
    label_file = os.path.join(base_path, f"{run}_Action label.txt")
    event_file = os.path.join(base_path, f"{run}_Event timestamp.txt")
    json_file = os.path.join(base_path, f"{run}_Session setup.json")

    # Kiểm tra đủ file
    if not all(os.path.exists(f) for f in [csv_file, label_file, event_file, json_file]):
        print(f"Missing file(s) for {run}, skipping...")
        continue

    eeg = pd.read_csv(csv_file, header=None).iloc[:, :22].values
    labels = [line.strip() for line in open(label_file)]
    events = [int(line.strip()) for line in open(event_file)]
    session_info = json.load(open(json_file))

    # === Step 1: Bandpass filter (8–30 Hz) ===
    eeg_filtered = bandpass_filter(eeg, low=8, high=30, fs=fs)

    # === Step 2: ICA artifact removal ===
    eeg_ica = apply_ica(eeg_filtered)

    # === Step 3: Normalization ===
    eeg_norm = normalize(eeg_ica)

    # === Step 4: Save cleaned full EEG ===
    save_path = os.path.join(output_dir, f"{run}_cleaned_full.npy")
    np.save(save_path, {
        "eeg_clean": eeg_norm,
        "labels": labels,
        "events": events,
        "sampling_rate": fs,
        "channel_count": eeg_norm.shape[1]
    })

    print(f"Saved cleaned EEG for {run}: {save_path}")
    print(f"   → Shape: {eeg_norm.shape}, Labels: {len(labels)}, Events: {len(events)}")

print("\n All runs processed and saved successfully!")



🧠 Processing Run1 ...
Saved cleaned EEG for Run1: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_full_runs/Run1_cleaned_full.npy
   → Shape: (12289, 22), Labels: 36, Events: 12

🧠 Processing Run2 ...
Saved cleaned EEG for Run2: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_full_runs/Run2_cleaned_full.npy
   → Shape: (12289, 22), Labels: 36, Events: 12

🧠 Processing Run3 ...
Saved cleaned EEG for Run3: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_full_runs/Run3_cleaned_full.npy
   → Shape: (12289, 22), Labels: 36, Events: 11

🧠 Processing Run4 ...
Saved cleaned EEG for Run4: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_full_runs/Run4_cleaned_full.npy
   → Shape: (12289, 22), Labels: 36, Events: 11

🧠 Processing Run5 ...
Saved cleaned EEG for Run5: /content/drive/MyDrive/truc/Data_UET175/ID01/S01_11_12_2024/cleaned_full_runs/Run5_cleaned_full.npy
   → Shape: (12289, 22), Labels: 36, Events: 12

🧠 Pr